# **Passe com sucesso ou sem sucesso**
  

# Deteção de passes e classificação do sucesso

**Objetivo:** detetar automaticamente os passes nos vídeos da Liga Portugal e classificar cada um como **bem-sucedido** (recebido por um colega) ou **falhado** (intercetado pelo adversário ou perdido).

**Definição usada:**
- **Posse de bola:** o jogador cujo pé está mais próximo da bola, durante pelo menos alguns frames consecutivos.
- **Passe:** mudança de posse do jogador A para o jogador B.
  - B da mesma equipa → ✅ passe bem-sucedido
  - B da equipa adversária → ❌ passe falhado (interceção)
  - bola sem novo dono / fora do campo → ❌ passe falhado

**Ferramentas:**
- **YOLOv8** (modelo treinado no Passo 2): deteção de jogadores, guarda-redes, árbitros e bola (*bounding boxes*).
- **Tracking (BoT-SORT):** identidade de cada jogador ao longo do tempo.
- **K-Means (CIELAB):** equipa de cada jogador.
- **MediaPipe Pose:** pontos do corpo de cada jogador, em particular os **pés**, para saber com precisão quem toca na bola.

## 1. Preparação

In [ ]:
%pip install -q "mediapipe==0.10.14" scikit-learn lap

from pathlib import Path
from ultralytics import YOLO
import cv2, numpy as np, matplotlib.pyplot as plt
import mediapipe as mp

BASE = Path(r"C:\Users\simao\OneDrive\Documentos\TDIA\3 ano\1_Semestre\PAIA-Projeto Aplicado de Inteligência Artificial I\OneDrive_2026-09-25\FOOTAR_DETECAO_DE_EVENTOS")
CAM_JOG = BASE / "treinos" / "players_ball_v1" / "weights" / "jogadores_best.pt"
modelo = YOLO(str(CAM_JOG))
BOLA, GR, JOG, ARB = 0, 1, 2, 3

videos = sorted((BASE / "videos").glob("*.mp4"))
PASTA_SAIDA = BASE / "resultados" / "passes"; PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print("MediaPipe versão:", mp.__version__)
print("Modelo:", modelo.names)
print(f"{len(videos)} vídeos:")
for v in videos:
    print("  -", v.name)

## 1.1 Teste do MediaPipe Pose num jogador
O MediaPipe Pose deteta 33 pontos do corpo humano numa imagem com **uma pessoa**. Por isso, aplica-se a cada jogador separadamente:
1. recorta-se o jogador a partir da *bounding box* do YOLO (com margem);
2. amplia-se o recorte, porque os jogadores numa transmissão têm poucos píxeis;
3. o MediaPipe deteta os pontos do corpo;
4. guardam-se os pontos dos **pés** (tornozelos 27/28 e pontas dos pés 31/32) e convertem-se de volta para coordenadas do frame.

Para este teste escolhe-se o jogador **mais próximo da bola**.

In [ ]:
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils
pose = mp_pose.Pose(static_image_mode=True, model_complexity=1, min_detection_confidence=0.3)
PES = {"tornozelo_esq": 27, "tornozelo_dir": 28, "pe_esq": 31, "pe_dir": 32}

def pes_jogador(frame, caixa, margem=0.15, altura=256):
    """Devolve as posições (no frame) dos pés de um jogador e o recorte desenhado."""
    x1, y1, x2, y2 = caixa.astype(int)
    w, h = x2 - x1, y2 - y1
    x1m, y1m = max(0, x1 - int(margem*w)), max(0, y1 - int(margem*h))
    x2m, y2m = min(frame.shape[1], x2 + int(margem*w)), min(frame.shape[0], y2 + int(margem*h))
    crop = frame[y1m:y2m, x1m:x2m]
    if crop.size == 0:
        return None, None
    esc = altura / crop.shape[0]
    grande = cv2.resize(crop, None, fx=esc, fy=esc, interpolation=cv2.INTER_CUBIC)
    res = pose.process(cv2.cvtColor(grande, cv2.COLOR_BGR2RGB))
    if not res.pose_landmarks:
        return None, grande
    lm = res.pose_landmarks.landmark
    gh, gw = grande.shape[:2]
    pes = {nome: (x1m + lm[i].x * gw / esc, y1m + lm[i].y * gh / esc, lm[i].visibility) for nome, i in PES.items()}
    desenho = grande.copy()
    mp_draw.draw_landmarks(desenho, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)
    return pes, desenho

# frame de teste: meio do primeiro vídeo
video = videos[0]
cap = cv2.VideoCapture(str(video))
cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2)
ok, frame = cap.read(); cap.release()

r = modelo(frame, conf=0.3, imgsz=1280, verbose=False)[0]
caixas = r.boxes.xyxy.cpu().numpy(); cls = r.boxes.cls.cpu().numpy().astype(int); conf = r.boxes.conf.cpu().numpy()
jogadores = caixas[cls == JOG]
bolas = caixas[cls == BOLA]

# jogador mais próximo da bola (ou o maior, se não houver bola)
if len(bolas):
    b = bolas[np.argmax(conf[cls == BOLA])]; bc = np.array([(b[0]+b[2])/2, (b[1]+b[3])/2])
    pes_base = np.column_stack([(jogadores[:, 0]+jogadores[:, 2])/2, jogadores[:, 3]])
    alvo = jogadores[np.argmin(np.linalg.norm(pes_base - bc, axis=1))]
else:
    alvo = jogadores[np.argmax((jogadores[:, 3]-jogadores[:, 1]))]

pes, desenho = pes_jogador(frame, alvo)
print("Pés detetados:" if pes else "MediaPipe não detetou o corpo neste jogador.")
if pes:
    for nome, (x, y, vis) in pes.items():
        print(f"  {nome:<14} x={x:6.0f}  y={y:6.0f}  visibilidade={vis:.2f}")

# visualização: recorte com esqueleto + frame com os pés marcados
img = frame.copy()
x1, y1, x2, y2 = alvo.astype(int)
cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), 2)
if pes:
    for x, y, _ in pes.values():
        cv2.circle(img, (int(x), int(y)), 5, (0, 0, 255), -1)
if len(bolas):
    cv2.circle(img, tuple(bc.astype(int)), 10, (255, 255, 255), 2)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={"width_ratios": [1, 3]})
if desenho is not None:
    a1.imshow(cv2.cvtColor(desenho, cv2.COLOR_BGR2RGB))
a1.set_title("MediaPipe Pose no recorte"); a1.axis("off")
a2.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); a2.set_title("Pés (vermelho) e bola (branco) no frame"); a2.axis("off")
plt.tight_layout(); plt.show()

## 1.1 Teste do MediaPipe Pose num jogador
O MediaPipe Pose deteta 33 pontos do corpo humano numa imagem com **uma pessoa**. Por isso, aplica-se a cada jogador separadamente:
1. recorta-se o jogador a partir da *bounding box* do YOLO (com margem);
2. amplia-se o recorte, porque os jogadores numa transmissão têm poucos píxeis;
3. o MediaPipe deteta os pontos do corpo;
4. guardam-se os pontos dos **pés** (tornozelos 27/28 e pontas dos pés 31/32) e convertem-se de volta para coordenadas do frame.

Para este teste escolhe-se o jogador **mais próximo da bola**.

In [ ]:
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils
pose = mp_pose.Pose(static_image_mode=True, model_complexity=1, min_detection_confidence=0.3)
PES = {"tornozelo_esq": 27, "tornozelo_dir": 28, "pe_esq": 31, "pe_dir": 32}

def pes_jogador(frame, caixa, margem=0.15, altura=256):
    """Devolve as posições (no frame) dos pés de um jogador e o recorte com o esqueleto."""
    x1, y1, x2, y2 = caixa.astype(int)
    w, h = x2 - x1, y2 - y1
    x1m, y1m = max(0, x1 - int(margem*w)), max(0, y1 - int(margem*h))
    x2m, y2m = min(frame.shape[1], x2 + int(margem*w)), min(frame.shape[0], y2 + int(margem*h))
    crop = frame[y1m:y2m, x1m:x2m]
    if crop.size == 0:
        return None, None
    esc = altura / crop.shape[0]
    grande = cv2.resize(crop, None, fx=esc, fy=esc, interpolation=cv2.INTER_CUBIC)
    res = pose.process(cv2.cvtColor(grande, cv2.COLOR_BGR2RGB))
    if not res.pose_landmarks:
        return None, grande
    lm = res.pose_landmarks.landmark
    gh, gw = grande.shape[:2]
    pes = {n: (x1m + lm[i].x * gw / esc, y1m + lm[i].y * gh / esc, lm[i].visibility) for n, i in PES.items()}
    desenho = grande.copy()
    mp_draw.draw_landmarks(desenho, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)
    return pes, desenho

def ler_frame(video, posicao=0.5):
    cap = cv2.VideoCapture(str(video))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT) * posicao))
    ok, f = cap.read(); cap.release()
    return f if ok else None

def jogador_perto_da_bola(frame):
    r = modelo(frame, conf=0.3, imgsz=1280, verbose=False)[0]
    cx = r.boxes.xyxy.cpu().numpy(); cl = r.boxes.cls.cpu().numpy().astype(int); cf = r.boxes.conf.cpu().numpy()
    jog, bol = cx[cl == JOG], cx[cl == BOLA]
    if len(jog) == 0:
        return None, None
    if len(bol):
        b = bol[np.argmax(cf[cl == BOLA])]; bc = np.array([(b[0]+b[2])/2, (b[1]+b[3])/2])
        base = np.column_stack([(jog[:, 0]+jog[:, 2])/2, jog[:, 3]])
        return jog[np.argmin(np.linalg.norm(base - bc, axis=1))], bc
    return jog[np.argmax(jog[:, 3]-jog[:, 1])], None

# ESCOLHE O VÍDEO PELO NOME e o momento (0.0 = início, 0.5 = meio, 1.0 = fim)
NOME = "Porto"
POSICAO = 0.5

video = next(v for v in videos if NOME.lower() in v.name.lower())
frame = ler_frame(video, POSICAO)
alvo, bc = jogador_perto_da_bola(frame)
pes, desenho = pes_jogador(frame, alvo)

print("Vídeo:", video.name)
print("Pés detetados:" if pes else "MediaPipe não detetou o corpo neste jogador.")
if pes:
    for n, (x, y, vis) in pes.items():
        print(f"  {n:<14} x={x:6.0f}  y={y:6.0f}  visibilidade={vis:.2f}")

img = frame.copy()
x1, y1, x2, y2 = alvo.astype(int)
cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), 2)
if pes:
    for x, y, _ in pes.values():
        cv2.circle(img, (int(x), int(y)), 5, (0, 0, 255), -1)
if bc is not None:
    cv2.circle(img, tuple(bc.astype(int)), 10, (255, 255, 255), 2)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={"width_ratios": [1, 3]})
if desenho is not None:
    a1.imshow(cv2.cvtColor(desenho, cv2.COLOR_BGR2RGB))
a1.set_title("MediaPipe Pose no recorte"); a1.axis("off")
a2.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); a2.set_title(f"{video.name}: pés (vermelho) e bola (branco)"); a2.axis("off")
plt.tight_layout(); plt.show()

### 1.2 Teste do MediaPipe em todos os vídeos
Para cada vídeo, usa-se o frame do meio e o jogador mais próximo da bola, e regista-se se o MediaPipe conseguiu detetar os pés. Mede-se assim a robustez do MediaPipe em diferentes estádios, ângulos de câmara e distâncias.

In [ ]:
resultados = []
for v in videos:
    f = ler_frame(v, 0.5)
    if f is None:
        resultados.append((v.name, "não leu", None, None)); continue
    alvo, bc = jogador_perto_da_bola(f)
    if alvo is None:
        resultados.append((v.name, "sem jogadores", None, None)); continue
    pes, _ = pes_jogador(f, alvo)
    altura_px = int(alvo[3] - alvo[1])
    vis = np.mean([p[2] for p in pes.values()]) if pes else None
    resultados.append((v.name, "✓ pés detetados" if pes else "✗ falhou", altura_px, vis))

print(f"{'Vídeo':<40}{'Resultado':<18}{'Altura jogador (px)':>20}{'Visibilidade pés':>18}")
for nome, res, alt, vis in resultados:
    print(f"{nome:<40}{res:<18}{str(alt):>20}{(f'{vis:.2f}' if vis is not None else '-'):>18}")
ok = sum(r[1].startswith("✓") for r in resultados)
print(f"\nMediaPipe detetou os pés em {ok} de {len(resultados)} vídeos ({100*ok/len(resultados):.0f}%)")

### 1.3 Conclusões do teste e estratégia "MediaPipe + bounding box"
- O MediaPipe detetou os pés em 60% dos vídeos. Quando deteta, a visibilidade dos pés fica entre 0.76 e 0.99.
- As falhas não dependem só do tamanho do jogador: a oclusão por outros jogadores, as costas viradas para a câmara e o desfoque de movimento também contam.
- **Estratégia em cascata:**
  1. MediaPipe normal (recorte com 256 px de altura);
  2. MediaPipe com recorte maior (384 px) e mais margem;
  3. MediaPipe "heavy" (`model_complexity=2`), mais lento mas mais robusto;
  4. Se todas falharem: **base da bounding box** (centro inferior da caixa YOLO) como aproximação dos pés.
- Regista-se o método usado em cada caso, para medir quanto o MediaPipe acrescenta em relação à bounding box.

In [ ]:
pose_forte = mp_pose.Pose(static_image_mode=True, model_complexity=2, min_detection_confidence=0.2)

def correr_pose(frame, caixa, modelo_pose, margem, altura):
    x1, y1, x2, y2 = caixa.astype(int)
    w, h = x2 - x1, y2 - y1
    x1m, y1m = max(0, x1 - int(margem*w)), max(0, y1 - int(margem*h))
    x2m, y2m = min(frame.shape[1], x2 + int(margem*w)), min(frame.shape[0], y2 + int(margem*h))
    crop = frame[y1m:y2m, x1m:x2m]
    if crop.size == 0:
        return None
    esc = altura / crop.shape[0]
    grande = cv2.resize(crop, None, fx=esc, fy=esc, interpolation=cv2.INTER_CUBIC)
    res = modelo_pose.process(cv2.cvtColor(grande, cv2.COLOR_BGR2RGB))
    if not res.pose_landmarks:
        return None
    lm = res.pose_landmarks.landmark
    gh, gw = grande.shape[:2]
    return {n: (x1m + lm[i].x*gw/esc, y1m + lm[i].y*gh/esc, lm[i].visibility) for n, i in PES.items()}

TENTATIVAS = [("mediapipe_256",   pose,       0.15, 256),
              ("mediapipe_384",   pose,       0.30, 384),
              ("mediapipe_heavy", pose_forte, 0.30, 384)]

def pes_robusto(frame, caixa):
    """Devolve (pes, metodo). Tenta o MediaPipe em cascata e, se falhar, usa a base da bounding box."""
    for nome, modelo_pose, margem, altura in TENTATIVAS:
        pes = correr_pose(frame, caixa, modelo_pose, margem, altura)
        if pes:
            return pes, nome
    x1, y1, x2, y2 = caixa
    return {"base_caixa": ((x1+x2)/2, y2, None)}, "bounding_box"

# Teste em todos os vídeos
from collections import Counter
contagem = Counter()
print(f"{'Vídeo':<40}{'Método usado':<18}{'Altura (px)':>12}")
for v in videos:
    f = ler_frame(v, 0.5)
    if f is None:
        continue
    alvo, _ = jogador_perto_da_bola(f)
    if alvo is None:
        continue
    pes, metodo = pes_robusto(f, alvo)
    contagem[metodo] += 1
    print(f"{v.name:<40}{metodo:<18}{int(alvo[3]-alvo[1]):>12}")

total = sum(contagem.values())
print("\nResumo:")
for m, n in contagem.most_common():
    print(f"  {m:<18} {n:>3}  ({100*n/total:.0f}%)")
mp_total = sum(n for m, n in contagem.items() if m.startswith("mediapipe"))
print(f"\nMediaPipe (qualquer tentativa): {mp_total}/{total} ({100*mp_total/total:.0f}%)  |  restantes com bounding box")

### 1.4 Resultado da cascata e verificação dos vídeos
- A cascata MediaPipe (256 → 384 → heavy) subiu a deteção dos pés de 60% para 77%. Nos restantes 23% usa-se a base da bounding box.
- Antes de avançar, verifica-se se todos os vídeos da pasta `videos` (incluindo os novos) abrem corretamente, têm o nome no formato `J<jornada>-<Casa>-<Fora>_<golos>-<golos>.mp4` e que resolução, fps e duração têm.(adição dos videos dA EPOCA 26/27)

In [ ]:
import re, pandas as pd

PADRAO = re.compile(r"^J\d+-[A-Za-z]+-[A-Za-z]+_\d+-\d+\.mp4$")

# Avisa se houver vídeos noutros formatos (que o glob("*.mp4") não apanha)
outros = [p.name for p in (BASE / "videos").iterdir() if p.suffix.lower() in {".mov", ".mkv", ".avi", ".m4v"}]
if outros:
    print("⚠ Vídeos noutro formato (converter para .mp4):", outros)

videos = sorted((BASE / "videos").glob("*.mp4"))
linhas = []
for v in videos:
    cap = cv2.VideoCapture(str(v))
    ok, _ = cap.read()
    fps = cap.get(cv2.CAP_PROP_FPS); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    linhas.append(dict(video=v.name, abre="✓" if ok else "✗",
                       nome_ok="✓" if PADRAO.match(v.name) else "✗ renomear",
                       resolucao=f"{w}x{h}", fps=round(fps, 1),
                       duracao_s=round(n / fps, 1) if fps else 0,
                       tamanho_MB=round(v.stat().st_size / 1e6, 1)))

df_videos = pd.DataFrame(linhas)
print(f"{len(df_videos)} vídeos | {(df_videos.abre=='✓').sum()} abrem | "
      f"{(df_videos.nome_ok=='✓').sum()} com nome correto | "
      f"duração total {df_videos.duracao_s.sum()/60:.1f} min")
problemas = df_videos[(df_videos.abre != "✓") | (df_videos.nome_ok != "✓") | (df_videos.resolucao != "1920x1080")]
print("\nVídeos com problemas:" if len(problemas) else "\nTodos os vídeos estão OK ✓")
display(problemas if len(problemas) else df_videos.head())

### 1.5 Organização final dos vídeos
- Os vídeos `_angle2` são o mesmo lance visto de outra câmara e mantêm-se, porque o padrão do nome passa a aceitar o sufixo `_angle<n>`.
- Os vídeos com `(1)` são comparados por hash (MD5) com o original. Os duplicados vão para `_duplicados` e os que forem diferentes são renomeados com o sufixo `_b`.
- As subpastas `penalties` e `uncut` ficam fora da análise de passes. Os penáltis serão usados mais tarde, na deteção desse evento.
- Os vídeos têm 50 fps. Na análise usa-se 1 frame em cada 2 (25 fps efetivos).

In [ ]:
import hashlib, shutil

def md5(p, bloco=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(bloco), b""):
            h.update(b)
    return h.hexdigest()

PASTA_V = BASE / "videos"
DUP = PASTA_V / "_duplicados"

for v in sorted(PASTA_V.glob("* (1).mp4")):
    original = v.with_name(v.name.replace(" (1)", ""))
    if original.exists() and md5(v) == md5(original):
        DUP.mkdir(exist_ok=True)
        shutil.move(str(v), DUP / v.name)
        print(f"Duplicado → _duplicados : {v.name}")
    else:
        novo = v.with_name(v.name.replace(" (1)", "").replace(".mp4", "_b.mp4"))
        v.rename(novo)
        print(f"Diferente → renomeado   : {v.name}  →  {novo.name}")

# Novo padrão: aceita _angle2 e _b
PADRAO = re.compile(r"^J\d+-[A-Za-z]+-[A-Za-z]+_\d+-\d+(_angle\d+|_b)?\.mp4$")
videos = sorted(PASTA_V.glob("*.mp4"))
mal = [v.name for v in videos if not PADRAO.match(v.name)]
print(f"\n{len(videos)} vídeos para análise | nome fora do padrão: {mal if mal else 'nenhum ✓'}")

for sub in ["penalties", "uncut"]:
    p = PASTA_V / sub
    if p.exists():
        n = list(p.rglob("*.mp4"))
        print(f"Subpasta {sub:<10}: {len(n)} vídeos, {sum(f.stat().st_size for f in n)/1e9:.2f} GB")

## Extração frame a frame de um lance de ataque
Fonte: pasta `videos/uncut` (185 lances de ataque, com sequências de passes e perdas de bola).

Para cada frame analisado (1 em cada 2, ou seja 25 fps efetivos):
1. **YOLO + ByteTrack:** jogadores, guarda-redes, árbitros e bola, cada um com ID persistente.
2. **Bola:** um único candidato por frame, escolhido pela proximidade à posição anterior (salto máximo de 150 px). Ignoram-se as zonas gráficas (placar e logótipo). Se a bola se perder durante mais de 10 frames, volta-se à maior confiança.
3. **Pés:** para os 3 jogadores mais próximos da bola, usa-se a cascata MediaPipe (256 → 384 → heavy), com a bounding box como último recurso.
4. **Equipas:** cor mediana do tronco em CIELAB (sem relva) + KMeans k=2 no clip todo + voto maioritário por ID. Equipa 1 = mais escura, 0 = mais clara, -1 = guarda-redes/árbitro/sem cor.

Resultado: duas tabelas, `jogadores` (uma linha por pessoa e frame) e `bola` (uma linha por frame), guardadas em CSV.

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from tqdm.auto import tqdm

PASSO, SALTO_MAX, N_PES = 2, 150, 3

def zona_grafica(x, y, W, H):
    return (x < 0.2*W and y < 0.2*H) or (x > 0.8*W and y > 0.75*H)

def cor_tronco(frame, caixa):
    x1, y1, x2, y2 = caixa.astype(int); w, h = x2 - x1, y2 - y1
    t = frame[y1 + int(.20*h): y1 + int(.45*h), x1 + int(.25*w): x2 - int(.25*w)]
    if t.size == 0:
        return None
    relva = cv2.inRange(cv2.cvtColor(t, cv2.COLOR_BGR2HSV), (35, 40, 40), (85, 255, 255)).reshape(-1)
    lab = cv2.cvtColor(t, cv2.COLOR_BGR2LAB).reshape(-1, 3)[relva == 0]
    return np.median(lab, axis=0) if len(lab) > 10 else None

def extrair_clip(video, max_seg=None, imgsz=1280):
    m = YOLO(str(CAM_JOG))                         # modelo novo = tracker limpo para cada vídeo
    cap = cv2.VideoCapture(str(video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_seg:
        total = min(total, int(max_seg * fps))
    jog, bola, ultima, perdida = [], [], None, 0

    for idx in tqdm(range(total), desc=video.name):
        ok, frame = cap.read()
        if not ok:
            break
        if idx % PASSO:
            continue
        r = m.track(frame, persist=True, tracker="bytetrack.yaml", conf=0.2, imgsz=imgsz, verbose=False)[0]
        cx = r.boxes.xyxy.cpu().numpy(); cl = r.boxes.cls.cpu().numpy().astype(int); cf = r.boxes.conf.cpu().numpy()
        ids = r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None else -np.ones(len(cl), int)

        # --- bola: 1 candidato por frame ---
        cand = [((b[0]+b[2])/2, (b[1]+b[3])/2, c) for b, c in zip(cx[cl == BOLA], cf[cl == BOLA])]
        cand = [c for c in cand if not zona_grafica(c[0], c[1], W, H)]
        bc = None
        if cand:
            if ultima is None:
                bc = max(cand, key=lambda c: c[2])
            else:
                d = [np.hypot(c[0]-ultima[0], c[1]-ultima[1]) for c in cand]
                i = int(np.argmin(d))
                if d[i] <= SALTO_MAX * (1 + perdida):
                    bc = cand[i]
        if bc:
            ultima, perdida = bc[:2], 0
        else:
            perdida += 1
            if perdida > 10:
                ultima = None
        bola.append(dict(frame=idx, t=idx/fps, x=bc[0] if bc else np.nan,
                         y=bc[1] if bc else np.nan, conf=bc[2] if bc else np.nan))

        # --- pessoas ---
        pessoas = np.where(cl != BOLA)[0]
        perto = set()
        if bc:
            pj = [i for i in pessoas if cl[i] in (JOG, GR)]
            d = [np.hypot((cx[i, 0]+cx[i, 2])/2 - bc[0], cx[i, 3] - bc[1]) for i in pj]
            perto = {pj[k] for k in np.argsort(d)[:N_PES]}
        for i in pessoas:
            lin = dict(frame=idx, t=idx/fps, id=ids[i], cls=modelo.names[cl[i]], conf=cf[i],
                       x1=cx[i, 0], y1=cx[i, 1], x2=cx[i, 2], y2=cx[i, 3],
                       cor=cor_tronco(frame, cx[i]) if cl[i] == JOG else None,
                       pe_esq_x=np.nan, pe_esq_y=np.nan, pe_dir_x=np.nan, pe_dir_y=np.nan, metodo_pes=None)
            if i in perto:
                pes, met = pes_robusto(frame, cx[i])
                if met == "bounding_box":
                    bx, by, _ = pes["base_caixa"]
                    lin.update(pe_esq_x=bx, pe_esq_y=by, pe_dir_x=bx, pe_dir_y=by)
                else:
                    lin.update(pe_esq_x=pes["pe_esq"][0], pe_esq_y=pes["pe_esq"][1],
                               pe_dir_x=pes["pe_dir"][0], pe_dir_y=pes["pe_dir"][1])
                lin["metodo_pes"] = met
            jog.append(lin)
    cap.release()

    df_j, df_b = pd.DataFrame(jog), pd.DataFrame(bola)

    # --- equipas: KMeans k=2 no clip todo + voto por ID ---
    df_j["equipa"] = -1
    mask = df_j.cor.notna()
    if mask.sum() >= 2:
        X = np.stack(df_j.loc[mask, "cor"].values)
        km = KMeans(2, n_init=10, random_state=0).fit(X)
        escura = int(np.argmin(km.cluster_centers_[:, 0]))
        df_j.loc[mask, "equipa"] = (km.labels_ == escura).astype(int)
        voto = df_j[mask & (df_j.id >= 0)].groupby("id").equipa.agg(lambda s: s.mode().iloc[0])
        com_id = mask & (df_j.id >= 0)
        df_j.loc[com_id, "equipa"] = df_j.loc[com_id, "id"].map(voto)
    df_j = df_j.drop(columns="cor")
    return df_j, df_b, fps

print("Funções do Proximo passo prontas ✓")

In [ ]:
uncut = sorted((BASE / "videos" / "uncut").rglob("*.mp4"))
print(f"{len(uncut)} lances em uncut. Primeiros: {[v.name for v in uncut[:5]]}")

# ⬇️ escolhe o lance pelo nome; MAX_SEG limita a duração para testar no PC (None = lance inteiro)
NOME, MAX_SEG = "Sporting", 8

video = next((v for v in uncut if NOME.lower() in v.name.lower()), uncut[0])
df_j, df_b, fps = extrair_clip(video, max_seg=MAX_SEG)

df_j.to_csv(PASTA_SAIDA / f"{video.stem}_jogadores.csv", index=False)
df_b.to_csv(PASTA_SAIDA / f"{video.stem}_bola.csv", index=False)

print(f"\nVídeo: {video.name}")
print(f"Frames analisados : {len(df_b)}")
print(f"Bola encontrada   : {df_b.x.notna().mean()*100:.0f}% dos frames")
print(f"IDs diferentes    : {df_j.id.nunique()}")
print("Pessoas por classe:", df_j.groupby('cls').id.nunique().to_dict())
print("Jogadores por equipa (IDs):", df_j[df_j.cls == 'player'].groupby('equipa').id.nunique().to_dict())
print("Método dos pés    :", df_j.metodo_pes.value_counts().to_dict())

# --- Figura: frame com bola + trajetória da bola ---
f_ok = df_j[df_j.metodo_pes.notna()].frame
fr = int(f_ok.iloc[len(f_ok)//2]) if len(f_ok) else int(df_b.frame.iloc[0])
cap = cv2.VideoCapture(str(video)); cap.set(cv2.CAP_PROP_POS_FRAMES, fr); _, img = cap.read(); cap.release()

CORES = {0: (0, 0, 255), 1: (255, 0, 0), -1: (0, 255, 255)}   # clara=vermelho, escura=azul, outros=amarelo
for _, p in df_j[df_j.frame == fr].iterrows():
    c = CORES[int(p.equipa)]
    cv2.rectangle(img, (int(p.x1), int(p.y1)), (int(p.x2), int(p.y2)), c, 2)
    cv2.putText(img, f"{p.id}", (int(p.x1), int(p.y1) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, c, 2)
    if pd.notna(p.pe_esq_x):
        for px, py in [(p.pe_esq_x, p.pe_esq_y), (p.pe_dir_x, p.pe_dir_y)]:
            cv2.circle(img, (int(px), int(py)), 6, (0, 255, 0), -1)
b = df_b[df_b.frame == fr].iloc[0]
if pd.notna(b.x):
    cv2.circle(img, (int(b.x), int(b.y)), 12, (255, 255, 255), 3)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(20, 7), gridspec_kw={"width_ratios": [2, 1]})
a1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); a1.axis("off")
a1.set_title(f"Frame {fr}: equipa clara (vermelho), escura (azul), outros (amarelo), pés (verde), bola (branco)")
a2.plot(df_b.t, df_b.x, ".-", label="x da bola"); a2.plot(df_b.t, df_b.y, ".-", label="y da bola")
a2.invert_yaxis(); a2.set_xlabel("tempo (s)"); a2.set_ylabel("píxeis"); a2.legend()
a2.set_title("Trajetória da bola (falhas = buracos)")
plt.tight_layout(); plt.show()

## Passo 3 — Interpolação da bola e posse de bola
**3.1 Interpolação:** só se preenchem buracos curtos (≤ 12 frames analisados ≈ 0.5 s), com interpolação linear. Buracos longos ficam vazios, porque normalmente correspondem à bola em voo num passe e inventar posições aí seria errado.

**3.2 Posse por frame:** o dono da bola é o jogador (ou GR) cujo pé está mais perto da bola, desde que:
- distância pé–bola ≤ 0.6 × altura do jogador em píxeis (o limiar normaliza a perspetiva, porque jogadores ao fundo são mais pequenos);
- ponto do pé = MediaPipe (pé esquerdo/direito) quando existe, senão a base da bounding box.

**3.3 Segmentos de posse:** posse consecutiva do mesmo ID durante ≥ 3 frames (≈ 0.12 s), ignorando os frames sem dono pelo meio. Elimina toques falsos de jogadores que só passam perto da bola.

In [ ]:
MAX_BURACO, FATOR_DIST, MIN_POSSE = 12, 0.6, 3

def interpolar_bola(df_b):
    b = df_b.copy()
    nan = b.x.isna()
    grupo = (nan != nan.shift()).cumsum()
    tam = nan.groupby(grupo).transform("sum")
    cheio = b[["x", "y"]].interpolate(limit_area="inside")
    preencher = nan & (tam <= MAX_BURACO) & cheio.x.notna()
    b.loc[preencher, ["x", "y"]] = cheio.loc[preencher]
    b["interpolada"] = preencher
    return b

def posse_por_frame(df_j, df_b):
    pj = df_j[df_j.cls.isin(["player", "goalkeeper"])]
    linhas = []
    for _, b in df_b.iterrows():
        lin = dict(frame=b.frame, t=b.t, dono=-1, equipa=-1, dist_rel=np.nan)
        p = pj[pj.frame == b.frame]
        if pd.notna(b.x) and len(p):
            cxb = (p.x1 + p.x2) / 2
            d_esq = np.hypot(p.pe_esq_x.fillna(cxb) - b.x, p.pe_esq_y.fillna(p.y2) - b.y)
            d_dir = np.hypot(p.pe_dir_x.fillna(cxb) - b.x, p.pe_dir_y.fillna(p.y2) - b.y)
            rel = (np.minimum(d_esq, d_dir) / (p.y2 - p.y1)).values
            k = int(np.argmin(rel))
            lin["dist_rel"] = rel[k]
            if rel[k] <= FATOR_DIST:
                lin["dono"], lin["equipa"] = int(p.id.iloc[k]), int(p.equipa.iloc[k])
        linhas.append(lin)
    return pd.DataFrame(linhas)

def segmentos_posse(df_p):
    com = df_p[df_p.dono >= 0].copy()
    com["seg"] = (com.dono != com.dono.shift()).cumsum()
    seg = com.groupby("seg").agg(dono=("dono", "first"), equipa=("equipa", "first"),
                                 t_ini=("t", "min"), t_fim=("t", "max"), n=("frame", "size")).reset_index(drop=True)
    return seg[seg.n >= MIN_POSSE].reset_index(drop=True)

df_bi = interpolar_bola(df_b)
df_p = posse_por_frame(df_j, df_bi)
seg = segmentos_posse(df_p)

print(f"Bola detetada   : {df_b.x.notna().mean()*100:.0f}%  →  com interpolação: {df_bi.x.notna().mean()*100:.0f}%")
print(f"Frames com dono : {(df_p.dono >= 0).mean()*100:.0f}%")
print(f"\nSegmentos de posse ({len(seg)}):")
display(seg)

# --- Linha temporal ---
fig, (a1, a2) = plt.subplots(2, 1, figsize=(16, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
det, interp = df_bi[~df_bi.interpolada], df_bi[df_bi.interpolada]
a1.plot(det.t, det.x, "b.", label="x detetado"); a1.plot(interp.t, interp.x, "c.", label="x interpolado")
a1.set_ylabel("x da bola (px)"); a1.legend(); a1.set_title("Bola e posse ao longo do tempo")
COR_EQ = {0: "red", 1: "blue", -1: "gold"}
for _, s in seg.iterrows():
    a2.barh(0, s.t_fim - s.t_ini + 0.04, left=s.t_ini, color=COR_EQ[s.equipa], edgecolor="k")
    a2.text((s.t_ini + s.t_fim) / 2, 0, str(s.dono), ha="center", va="center", color="white", fontweight="bold")
a2.set_yticks([]); a2.set_xlabel("tempo (s)")
a2.set_title("Posse: equipa clara (vermelho), escura (azul), GR/sem equipa (amarelo); número = ID")
plt.tight_layout(); plt.show()

### 3.4 Validação com o que acontece no vídeo (ground truth)
**Vídeo:** `J1-EstrelaAmadora-Sporting_0-1` (uncut), primeiros 8 s.

| Momento | Realidade | Sistema |
|---|---|---|
| 0–3 s | Jogador do Sporting conduz a bola | ✓ Posse do ID 8 (0–3.08 s) |
| ~3 s | Passe para um colega | ✗ bola perdida |
| logo depois | Remate de primeira → golo | ✗ bola perdida |

**Diagnóstico:** a bola deixa de ser detetada exatamente quando começa a andar depressa (passe/remate), por causa do desfoque de movimento.

**Teste:** nos frames entre 3.0 e 4.6 s, corre-se o YOLO só para a classe bola com confiança mínima (0.01) em duas resoluções (1280 e 1920 = nativa), para perceber se o modelo vê a bola com confiança baixa ou se não a vê de todo.

In [ ]:
ult = df_b[df_b.x.notna() & (df_b.t <= 3.2)].iloc[-1]          # última posição conhecida da bola
cx0, cy0 = int(ult.x), int(ult.y)
print(f"Última bola vista: t={ult.t:.2f}s em ({cx0}, {cy0})\n")

T_INI, T_FIM = 3.0, 4.6
cap = cv2.VideoCapture(str(video)); fps = cap.get(cv2.CAP_PROP_FPS)
idxs = np.linspace(T_INI * fps, T_FIM * fps, 8).astype(int)

print(f"{'t (s)':>6} | {'conf máx 1280':>14} | {'conf máx 1920':>14} | candidatos>0.05 (1280/1920)")
fig, axs = plt.subplots(2, 4, figsize=(22, 8))
for ax, i in zip(axs.flat, idxs):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, f = cap.read()
    img, maxs, ncand = f.copy(), [], []
    for tam, cor in [(1280, (0, 255, 255)), (1920, (255, 0, 255))]:
        r = modelo(f, conf=0.01, imgsz=tam, classes=[BOLA], verbose=False)[0]
        b, c = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()
        maxs.append(c.max() if len(c) else 0); ncand.append(int((c > 0.05).sum()))
        for bb, cc in zip(b, c):
            if cc > 0.03:
                x1, y1, x2, y2 = bb.astype(int)
                cv2.rectangle(img, (x1 - 6, y1 - 6), (x2 + 6, y2 + 6), cor, 2)
                cv2.putText(img, f"{cc:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, cor, 2)
    print(f"{i/fps:6.2f} | {maxs[0]:14.2f} | {maxs[1]:14.2f} | {ncand[0]}/{ncand[1]}")
    x0, y0 = max(0, cx0 - 600), max(0, cy0 - 350)
    ax.imshow(cv2.cvtColor(img[y0:y0 + 700, x0:x0 + 1200], cv2.COLOR_BGR2RGB))
    ax.set_title(f"t={i/fps:.2f}s | 1280: {maxs[0]:.2f}  1920: {maxs[1]:.2f}"); ax.axis("off")
cap.release()
fig.suptitle("Candidatos a bola: amarelo = imgsz 1280, magenta = imgsz 1920 (recorte à volta da última bola vista)")
plt.tight_layout(); plt.show()

### 3.5 Diagnóstico: o modelo não vê a bola rápida
- Entre 3.2 e 4.1 s (passe + remate) a confiança máxima da bola fica entre 0.00 e 0.10, a 1280 e a 1920. Não é um problema de limiar: o modelo não reconhece a bola desfocada.
- A 1920 aparecem falsos positivos nas chuteiras/meias e o resultado não é melhor do que a 1280.
- **Causa:** domain gap. O dataset de treino é da Bundesliga (câmara tática) e quase não tem bola em movimento rápido.

**Teste seguinte, zoom por blocos (SAHI):** o frame é dividido em 3×2 blocos sobrepostos e o YOLO corre em cada bloco a imgsz 1280. A bola fica ~2× maior. Compara-se a confiança máxima com e sem blocos.

In [ ]:
def bola_por_blocos(frame, nx=3, ny=2, sobrep=0.15, conf=0.05, imgsz=1280):
    H, W = frame.shape[:2]
    bw, bh = int(W / nx * (1 + sobrep)), int(H / ny * (1 + sobrep))
    cands = []
    for ix in range(nx):
        for iy in range(ny):
            x0 = min(int(ix * W / nx), W - bw); y0 = min(int(iy * H / ny), H - bh)
            r = modelo(frame[y0:y0 + bh, x0:x0 + bw], conf=conf, imgsz=imgsz, classes=[BOLA], verbose=False)[0]
            for b, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                cands.append((b + [x0, y0, x0, y0], float(c)))
    return cands

cap = cv2.VideoCapture(str(video))
print(f"{'t (s)':>6} | {'sem blocos':>10} | {'com blocos':>10} | candidatos>0.1 com blocos")
fig, axs = plt.subplots(2, 4, figsize=(22, 8))
for ax, i in zip(axs.flat, idxs):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, f = cap.read()
    r = modelo(f, conf=0.01, imgsz=1280, classes=[BOLA], verbose=False)[0]
    c_normal = r.boxes.conf.max().item() if len(r.boxes) else 0
    cands = bola_por_blocos(f)
    c_blocos = max([c for _, c in cands], default=0)
    img = f.copy()
    for b, c in cands:
        if c > 0.1:
            x1, y1, x2, y2 = b.astype(int)
            cv2.rectangle(img, (x1 - 10, y1 - 10), (x2 + 10, y2 + 10), (0, 0, 255), 4)
            cv2.putText(img, f"{c:.2f}", (x1, y1 - 15), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    print(f"{i/fps:6.2f} | {c_normal:10.2f} | {c_blocos:10.2f} | {sum(c > 0.1 for _, c in cands)}")
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis("off")
    ax.set_title(f"t={i/fps:.2f}s | normal {c_normal:.2f} → blocos {c_blocos:.2f}")
cap.release()
fig.suptitle("Bola com zoom por blocos (vermelho = candidato com conf > 0.1)")
plt.tight_layout(); plt.show()

### 3.6 Resultado do zoom por blocos e análise do passe aos 2 s
**Ground truth:** passe aos ~2 s, remate de primeira aos ~3 s → golo.

- **Zoom por blocos:** a confiança sobe, mas os candidatos aparecem nas chuteiras/meias dos jogadores, não na trajetória do remate. Fica descartado como solução para a bola rápida, que exige um modelo de bola treinado em vídeos da FootAR.
- **Passe aos 2 s:** a bola foi detetada sem falhas de 0 a 3.08 s, mas o sistema atribuiu a posse sempre ao ID 8. O erro está na lógica de posse ou no tracking, não na deteção.

Para cada instante entre 1.0 e 3.0 s, mostram-se os 3 jogadores mais próximos da bola (ID, equipa, distância relativa) e o recorte da imagem à volta da bola.

In [ ]:
def mais_proximos(frame_idx, bx, by, k=3):
    p = df_j[(df_j.frame == frame_idx) & df_j.cls.isin(["player", "goalkeeper"])].copy()
    cxb = (p.x1 + p.x2) / 2
    d_esq = np.hypot(p.pe_esq_x.fillna(cxb) - bx, p.pe_esq_y.fillna(p.y2) - by)
    d_dir = np.hypot(p.pe_dir_x.fillna(cxb) - bx, p.pe_dir_y.fillna(p.y2) - by)
    p["dist_rel"] = np.minimum(d_esq, d_dir) / (p.y2 - p.y1)
    return p.sort_values("dist_rel").head(k)

TEMPOS = [1.0, 1.4, 1.8, 2.2, 2.6, 3.0]
cap = cv2.VideoCapture(str(video))
fig, axs = plt.subplots(2, 3, figsize=(21, 9))
for ax, t in zip(axs.flat, TEMPOS):
    b = df_bi.iloc[(df_bi.t - t).abs().argmin()]
    fr = int(b.frame)
    cap.set(cv2.CAP_PROP_POS_FRAMES, fr); _, img = cap.read()
    top = mais_proximos(fr, b.x, b.y)
    print(f"t={b.t:.2f}s  bola=({b.x:.0f},{b.y:.0f})  " +
          "  |  ".join(f"ID {int(r.id)} eq{int(r.equipa)} d={r.dist_rel:.2f}" for _, r in top.iterrows()))
    for n, (_, r) in enumerate(top.iterrows()):
        cor = (0, 255, 0) if n == 0 else (0, 165, 255)          # verde = mais próximo, laranja = 2.º e 3.º
        cv2.rectangle(img, (int(r.x1), int(r.y1)), (int(r.x2), int(r.y2)), cor, 2)
        cv2.putText(img, f"ID{int(r.id)} d={r.dist_rel:.2f}", (int(r.x1), int(r.y1) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, cor, 2)
        if pd.notna(r.pe_esq_x):
            for px, py in [(r.pe_esq_x, r.pe_esq_y), (r.pe_dir_x, r.pe_dir_y)]:
                cv2.circle(img, (int(px), int(py)), 4, (255, 0, 255), -1)
    cv2.circle(img, (int(b.x), int(b.y)), 10, (255, 255, 255), 2)
    x0, y0 = max(0, int(b.x) - 350), max(0, int(b.y) - 200)
    ax.imshow(cv2.cvtColor(img[y0:y0 + 400, x0:x0 + 700], cv2.COLOR_BGR2RGB)); ax.axis("off")
    ax.set_title(f"t={b.t:.2f}s | dono: ID {int(top.iloc[0].id)}" if len(top) else f"t={b.t:.2f}s")
cap.release()
fig.suptitle("Verde = jogador mais próximo da bola (dono), laranja = 2.º/3.º, magenta = pés, branco = bola")
plt.tight_layout(); plt.show()

### 3.7 Conclusão da análise do lance Estrela Amadora–Sporting
- **Tracking:** o ID 8 mantém-se estável durante toda a condução (sem trocas de ID).
- **Posse:** o dono fica a uma distância de 0.19–0.36 alturas e o 2.º mais próximo a ≥ 1.09. O limiar 0.6 separa bem os dois.
- **Passe + remate de primeira (~3.0–3.5 s no tempo do vídeo):** a bola é rápida, o modelo deixa de a detetar e não há receção visível.
- **Limitação identificada:** deteção de bola rápida. Solução: modelo de bola treinado em vídeos da FootAR (pedido à empresa).

## Passo 4 — Deteção de passes e classificação certo / falhado
A partir dos segmentos de posse (Passo 3), para cada par de segmentos consecutivos A → B:

| Situação | Classificação |
|---|---|
| Dono diferente, mesma equipa | **passe certo** |
| Dono diferente, equipa diferente | **passe falhado (interceção)** |
| Mesma equipa, IDs diferentes mas caixas quase no mesmo sítio (< 0.5 alturas) e intervalo < 0.3 s | **troca de ID**, não é passe |
| Intervalo entre A e B > 3 s | **indeterminado** (bola perdida durante muito tempo) |
| Última posse e a bola não reaparece | **indeterminado** (passe/remate não observado) |
| GR ou jogador sem equipa | **indeterminado** (equipa desconhecida) |

Junta-se tudo numa função `analisar_lance()` (extração → interpolação → posse → passes) para correr em vários lances.

In [ ]:
MAX_VOO, DIST_TROCA_ID = 3.0, 0.5

def detetar_passes(seg, df_j, df_b):
    passes = []
    for a, b in zip(seg.iloc[:-1].itertuples(), seg.iloc[1:].itertuples()):
        voo = b.t_ini - a.t_fim
        base = dict(t_passe=round(a.t_fim, 2), t_rececao=round(b.t_ini, 2), voo_s=round(voo, 2),
                    passador=a.dono, eq_passador=a.equipa, recetor=b.dono, eq_recetor=b.equipa)
        if a.dono == b.dono:
            continue
        if voo > MAX_VOO:
            passes.append({**base, "resultado": "indeterminado (bola perdida)"}); continue
        # troca de ID? (mesmo jogador com ID novo)
        ra = df_j[(df_j.id == a.dono) & np.isclose(df_j.t, a.t_fim)]
        rb = df_j[(df_j.id == b.dono) & np.isclose(df_j.t, b.t_ini)]
        if len(ra) and len(rb) and voo < 0.3 and a.equipa == b.equipa:
            ra, rb = ra.iloc[0], rb.iloc[0]
            d = np.hypot((ra.x1 + ra.x2)/2 - (rb.x1 + rb.x2)/2, ra.y2 - rb.y2) / (ra.y2 - ra.y1)
            if d < DIST_TROCA_ID:
                continue
        if a.equipa < 0 or b.equipa < 0:
            res = "indeterminado (equipa desconhecida)"
        elif a.equipa == b.equipa:
            res = "certo"
        else:
            res = "falhado (interceção)"
        passes.append({**base, "resultado": res})
    if len(seg):
        u = seg.iloc[-1]
        if df_b.t.max() - u.t_fim > 0.5:
            passes.append(dict(t_passe=round(u.t_fim, 2), t_rececao=None, voo_s=None, passador=u.dono,
                               eq_passador=u.equipa, recetor=None, eq_recetor=None,
                               resultado="indeterminado (bola não reaparece)"))
    return pd.DataFrame(passes)

def analisar_lance(video, max_seg=None):
    df_j, df_b, fps = extrair_clip(video, max_seg=max_seg)
    df_bi = interpolar_bola(df_b)
    seg = segmentos_posse(posse_por_frame(df_j, df_bi))
    ps = detetar_passes(seg, df_j, df_bi)
    for nome, df in [("jogadores", df_j), ("bola", df_bi), ("posse", seg), ("passes", ps)]:
        df.to_csv(PASTA_SAIDA / f"{video.stem}_{nome}.csv", index=False)
    return seg, ps, df_bi

# Teste no lance que já analisámos (usa os dados já extraídos, não volta a correr o YOLO)
ps = detetar_passes(seg, df_j, df_bi)
print("Lance Estrela Amadora–Sporting:")
display(ps)

In [ ]:
# ⬇️ escolhe 3 lances de jogos diferentes (podes trocar os nomes)
ESCOLHA, MAX_SEG = ["Benfica", "Porto", "Braga"], 10

lances = []
for k in ESCOLHA:
    v = next((v for v in uncut if k.lower() in v.name.lower()), None)
    if v and v not in lances:
        lances.append(v)

todos = []
for v in lances:
    seg_v, ps_v, bi_v = analisar_lance(v, max_seg=MAX_SEG)
    print(f"\n=== {v.name} | bola {bi_v.x.notna().mean()*100:.0f}% | {len(seg_v)} posses | {len(ps_v)} passes ===")
    display(seg_v); display(ps_v)
    if len(ps_v):
        todos.append(ps_v.assign(video=v.name))

if todos:
    resumo = pd.concat(todos)
    print("\nResumo geral:")
    print(resumo.resultado.value_counts().to_string())